In [ ]:
!pip -q install numpy scipy scikit-learn h5py onnx onnxscript onnxruntime skl2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.7 MB/s eta 0:00:00


In [ ]:
import os
import gc
import glob
import json
import time
import random

import h5py
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import onnxruntime as ort

from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "/content/drive/MyDrive/global_neurotech_hack_2026/medium_data"
DATASET_KEY = "acquisition/ElectricalSeries"

TOTAL_CHANNELS = 76
ENCODED_CHANNELS = 64
GT_CHANNELS = 12

FS = 32000
BIN_SIZE = 320          # 10 ms at 32 kHz
CONTEXT_BINS = 20       # 200 ms context
CHUNK_BINS = 256
FEATURE_MODE = "mean_rms"  # "mean" -> 1280 features, "mean_rms" -> 2560 features

CACHE_DIR = "/content/binned_cache"
OUT_DIR = "/content/decoder_outputs_ridge"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ── Glob both Ani and ksenija medium files ──
FILES = sorted(
    glob.glob(os.path.join(DATA_DIR, "smoke_Ani_seed*.h5")) +
    glob.glob(os.path.join(DATA_DIR, "medium-ksenija-smoke-*.h5"))
)
assert len(FILES) > 0, f"No .h5 files found in {DATA_DIR}"

# ── Verify every file: 76-channel layout + print n_frames ──
print("File verification:")
for fpath in FILES:
    with h5py.File(fpath, "r") as _f:
        _ds = _f[DATASET_KEY]
        _flat_len = int(_ds.shape[0])
        assert _flat_len % TOTAL_CHANNELS == 0, (
            f"{os.path.basename(fpath)}: flat_len={_flat_len} not divisible by {TOTAL_CHANNELS}"
        )
        _n_frames = _flat_len // TOTAL_CHANNELS
        print(f"  {os.path.basename(fpath):40s}  n_frames={_n_frames:>10d}")

# ── Verify sample rate from first file ──
with h5py.File(FILES[0], "r") as _f:
    _ds = _f[DATASET_KEY]
    _rate = _ds.attrs.get("rate", None)
    if _rate is not None:
        assert abs(float(_rate) - FS) < 1.0, \
            f"FS mismatch! File rate={_rate} Hz, expected FS={FS}. Update FS and BIN_SIZE."
        print(f"\nSample rate verified: {float(_rate)} Hz matches FS={FS}")
    else:
        print(f"\nNo rate attr on dataset; using FS={FS} (verify manually)")

FEATURE_DIM = CONTEXT_BINS * ENCODED_CHANNELS * (2 if FEATURE_MODE == "mean_rms" else 1)
print(f"Feature mode: {FEATURE_MODE}, feature dim: {FEATURE_DIM}")
print(f"Total files: {len(FILES)}")

In [ ]:
def bin_file_to_arrays(
    h5_path,
    dataset_key=DATASET_KEY,
    total_channels=TOTAL_CHANNELS,
    encoded_channels=ENCODED_CHANNELS,
    bin_size=BIN_SIZE,
    chunk_bins=CHUNK_BINS,
    cache_dir=CACHE_DIR,
):
    """
    Reads flattened ElectricalSeries, bins every bin_size raw frames.
    Computes both mean and RMS for encoded channels.
    Returns:
        Xb_mean: (n_bins, 64) float32
        Xb_rms:  (n_bins, 64) float32
        Yb:      (n_bins, 12) float32
    """
    stem = os.path.splitext(os.path.basename(h5_path))[0]
    cache_path = os.path.join(cache_dir, f"{stem}_bin{bin_size}_v2.npz")

    if os.path.exists(cache_path):
        z = np.load(cache_path)
        Xb_mean = z["X_mean"].astype(np.float32)
        Xb_rms = z["X_rms"].astype(np.float32)
        Yb = z["Y"].astype(np.float32)
        print(f"[cache] {stem}: X_mean={Xb_mean.shape}, X_rms={Xb_rms.shape}, Y={Yb.shape}")
        return Xb_mean, Xb_rms, Yb

    Xm_parts, Xr_parts, Y_parts = [], [], []

    with h5py.File(h5_path, "r") as f:
        dset = f[dataset_key]
        flat_len = int(dset.shape[0])

        assert flat_len % total_channels == 0, (
            f"{h5_path}: flat length {flat_len} not divisible by {total_channels}"
        )

        n_frames = flat_len // total_channels
        usable_frames = (n_frames // bin_size) * bin_size
        usable_values = usable_frames * total_channels
        values_per_chunk = chunk_bins * bin_size * total_channels

        print(f"[read ] {stem}: n_frames={n_frames}, usable_bins={usable_frames // bin_size}")

        for start in range(0, usable_values, values_per_chunk):
            end = min(start + values_per_chunk, usable_values)
            flat = np.asarray(dset[start:end], dtype=np.float32)
            frames = flat.reshape(-1, total_channels)
            reshaped = frames.reshape(-1, bin_size, total_channels)
            binned_mean = reshaped.mean(axis=1)
            binned_rms = np.sqrt((reshaped ** 2).mean(axis=1))

            Xm_parts.append(binned_mean[:, :encoded_channels].astype(np.float32))
            Xr_parts.append(binned_rms[:, :encoded_channels].astype(np.float32))
            Y_parts.append(binned_mean[:, encoded_channels:].astype(np.float32))

            del flat, frames, reshaped, binned_mean, binned_rms
            gc.collect()

    Xb_mean = np.concatenate(Xm_parts, axis=0).astype(np.float32)
    Xb_rms = np.concatenate(Xr_parts, axis=0).astype(np.float32)
    Yb = np.concatenate(Y_parts, axis=0).astype(np.float32)

    np.savez(cache_path, X_mean=Xb_mean, X_rms=Xb_rms, Y=Yb)
    print(f"[saved] {stem}: X_mean={Xb_mean.shape}, X_rms={Xb_rms.shape}, Y={Yb.shape}")

    return Xb_mean, Xb_rms, Yb


binned_files = []
for path in FILES:
    Xb_mean, Xb_rms, Yb = bin_file_to_arrays(path)
    binned_files.append((path, Xb_mean, Xb_rms, Yb))

print("\nSummary:")
for path, Xb_mean, Xb_rms, Yb in binned_files:
    print(os.path.basename(path), "X_mean:", Xb_mean.shape, "X_rms:", Xb_rms.shape, "Y:", Yb.shape)

In [ ]:
def make_windows(Xb_mean, Xb_rms, Yb, context_bins=CONTEXT_BINS, feature_mode=FEATURE_MODE):
    """
    Sliding context windows with optional RMS features.
    feature_mode:
      "mean"     -> Xw shape (N, context_bins*64)   = (N, 1280)
      "mean_rms" -> Xw shape (N, context_bins*128)  = (N, 2560)
    Returns:
        Xw: (N, D) features
        Y_scaled: (N, 10) ground truth scaled to controller ranges
    """
    T, C = Xb_mean.shape
    N = T - context_bins + 1
    if N <= 0:
        raise ValueError(f"Not enough bins: T={T}, context_bins={context_bins}")

    def _strided_windows(arr):
        shape = (N, context_bins, C)
        strides = (arr.strides[0], arr.strides[0], arr.strides[1])
        return np.lib.stride_tricks.as_strided(
            arr, shape=shape, strides=strides
        ).copy().reshape(N, context_bins * C).astype(np.float32)

    Xw = _strided_windows(Xb_mean)

    if feature_mode == "mean_rms":
        Xw = np.concatenate([Xw, _strided_windows(Xb_rms)], axis=1)

    # Predict only gt_0..gt_9; scale to controller ranges
    Y = Yb[context_bins - 1:, :10].astype(np.float32)
    Y_scaled = np.empty_like(Y, dtype=np.float32)
    Y_scaled[:, :4] = np.clip(Y[:, :4] / 32767.0, -1.0, 1.0)
    Y_scaled[:, 4:10] = np.clip(Y[:, 4:10] / 32767.0, 0.0, 1.0)

    return Xw, Y_scaled


def make_sample_weights(Y_scaled):
    """Emphasize active bins so the model doesn't collapse to near-zero predictions."""
    w = np.ones(len(Y_scaled), dtype=np.float32)
    analog_active = np.max(np.abs(Y_scaled[:, :4]), axis=1) > 0.10
    sparse_active = np.max(Y_scaled[:, 4:10], axis=1) > 0.05
    w += 1.0 * analog_active.astype(np.float32)
    w += 2.0 * sparse_active.astype(np.float32)
    return w

In [ ]:
# Hold out the last file as validation for alpha selection
train_indices = list(range(len(binned_files) - 1))
val_indices = [len(binned_files) - 1]

print("Train files:")
for i in train_indices:
    print(" -", os.path.basename(binned_files[i][0]))
print("Val file:")
for i in val_indices:
    print(" -", os.path.basename(binned_files[i][0]))


def build_split(indices):
    X_list, Y_list, W_list = [], [], []
    for i in indices:
        _, Xb_mean, Xb_rms, Yb = binned_files[i]
        Xw, Yw = make_windows(Xb_mean, Xb_rms, Yb)
        W = make_sample_weights(Yw)
        X_list.append(Xw.astype(np.float32))
        Y_list.append(Yw.astype(np.float32))
        W_list.append(W.astype(np.float32))
        del Xw, Yw, W
        gc.collect()
    X = np.concatenate(X_list, axis=0).astype(np.float32)
    Y = np.concatenate(Y_list, axis=0).astype(np.float32)
    W = np.concatenate(W_list, axis=0).astype(np.float32)
    return X, Y, W


X_train, Y_train, W_train = build_split(train_indices)
X_val, Y_val, W_val = build_split(val_indices)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def evaluate_predictions(y_true, y_pred):
    corrs = [safe_corr(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    maes = np.mean(np.abs(y_true - y_pred), axis=0)
    valid = np.array([not np.isnan(c) for c in corrs], dtype=bool)
    mean_corr = float(np.mean(np.array(corrs)[valid])) if valid.any() else np.nan
    return {"corrs": corrs, "mean_corr": mean_corr, "maes": maes}


alphas = [3.0, 10.0, 30.0, 100.0]
results = []

for alpha in alphas:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, fit_intercept=True))
    ])

    t0 = time.time()
    pipe.fit(X_train, Y_train, ridge__sample_weight=W_train)
    y_val_pred = pipe.predict(X_val).astype(np.float32)

    y_val_pred[:, :4] = np.clip(y_val_pred[:, :4], -1.0, 1.0)
    y_val_pred[:, 4:10] = np.clip(y_val_pred[:, 4:10], 0.0, 1.0)

    metrics = evaluate_predictions(Y_val, y_val_pred)
    dt = time.time() - t0

    results.append((alpha, metrics["mean_corr"], metrics["corrs"], metrics["maes"], y_val_pred))
    print(f"alpha={alpha:6.1f} | val_mean_corr={metrics['mean_corr']:.4f} | time={dt:.1f}s")

best_idx = max(range(len(results)), key=lambda i: results[i][1])
best_alpha = results[best_idx][0]
best_val_pred = results[best_idx][4]
print(f"\nBest alpha: {best_alpha}")

for alpha, mean_corr, corrs, maes, _ in results:
    print(f"\nalpha={alpha}")
    for i, c in enumerate(corrs):
        print(f"  gt_{i}: corr={c:.4f} mae={maes[i]:.4f}")

In [ ]:
# ── Proxy bitrate evaluation (Shenoy B) on val set ──
def proxy_bitrate(Y_true_scaled, Y_pred_scaled, N=8, mag_thresh=0.3, window_sec=0.2):
    """Proxy ITR from stick channels (0, 1) mapped to N angular symbols."""
    def to_symbol(y):
        x, yv = y[:, 0], y[:, 1]
        mag = np.sqrt(x**2 + yv**2)
        angle = np.arctan2(yv, x)
        sym = ((angle + np.pi) / (2 * np.pi) * N).astype(int) % N
        sym[mag < mag_thresh] = -1
        return sym

    s_true = to_symbol(Y_true_scaled)
    s_pred = to_symbol(Y_pred_scaled)

    mask = s_true >= 0
    pred_emit = s_pred >= 0
    Sc = int(((s_true == s_pred) & mask & pred_emit).sum())
    Si = int(((s_true != s_pred) & mask & pred_emit).sum())
    t = float(mask.sum()) * window_sec
    if t == 0:
        return 0.0
    return float(np.log2(max(N - 1, 1)) * max(Sc - Si, 0) / t)


print("Proxy bitrate table (stick channels 0,1 on val set)")
print(f"{'N':>3s}  {'mag_thresh':>10s}  {'bitrate':>8s}")
print(f"{'---':>3s}  {'----------':>10s}  {'--------':>8s}")
for N_sym in [4, 6, 8]:
    for mt in [0.15, 0.2, 0.25, 0.3, 0.35, 0.4]:
        br = proxy_bitrate(Y_val, best_val_pred, N=N_sym, mag_thresh=mt)
        print(f"{N_sym:>3d}  {mt:>10.2f}  {br:>8.4f}")

In [ ]:
# Free previous split arrays
del X_train, Y_train, W_train, X_val, Y_val, W_val, best_val_pred
gc.collect()

all_indices = list(range(len(binned_files)))
X_all, Y_all, W_all = build_split(all_indices)

print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)
print("W_all:", W_all.shape)

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_alpha, fit_intercept=True))
])

t0 = time.time()
final_pipe.fit(X_all, Y_all, ridge__sample_weight=W_all)
print(f"Final fit finished in {time.time() - t0:.1f}s")

# quick training-set sanity check
y_all_pred = final_pipe.predict(X_all).astype(np.float32)
y_all_pred[:, :4] = np.clip(y_all_pred[:, :4], -1.0, 1.0)
y_all_pred[:, 4:10] = np.clip(y_all_pred[:, 4:10], 0.0, 1.0)

train_metrics = evaluate_predictions(Y_all, y_all_pred)
print("Train mean corr:", train_metrics["mean_corr"])
for i, c in enumerate(train_metrics["corrs"]):
    print(f"  gt_{i}: corr={c:.4f} mae={train_metrics['maes'][i]:.4f}")

In [ ]:
skl_onnx_path = os.path.join(OUT_DIR, "model_sklearn.onnx")

initial_type = [("input", FloatTensorType([None, FEATURE_DIM]))]
onnx_model = to_onnx(final_pipe, initial_types=initial_type, target_opset=17)

with open(skl_onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Saved sklearn ONNX to:", skl_onnx_path)

In [ ]:
scaler = final_pipe.named_steps["scaler"]
ridge = final_pipe.named_steps["ridge"]

x_mean = scaler.mean_.astype(np.float32)
x_scale = scaler.scale_.astype(np.float32)
x_scale = np.where(x_scale < 1e-8, 1.0, x_scale).astype(np.float32)

W = ridge.coef_.astype(np.float32)        # shape: (10, FEATURE_DIM)
b = ridge.intercept_.astype(np.float32)   # shape: (10,)

print("x_mean:", x_mean.shape)
print("x_scale:", x_scale.shape)
print("W:", W.shape)
print("b:", b.shape)


class RidgeDecoderWrapper(nn.Module):
    def __init__(self, x_mean, x_scale, W, b):
        super().__init__()
        self.register_buffer("x_mean", torch.tensor(x_mean, dtype=torch.float32).view(1, -1))
        self.register_buffer("x_scale", torch.tensor(x_scale, dtype=torch.float32).view(1, -1))
        self.register_buffer("W", torch.tensor(W, dtype=torch.float32))
        self.register_buffer("b", torch.tensor(b, dtype=torch.float32).view(1, -1))

    def forward(self, x):
        # x: [batch, FEATURE_DIM]
        x = (x - self.x_mean) / self.x_scale
        y10 = x @ self.W.T + self.b

        y_signed = torch.clamp(y10[:, :4], -1.0, 1.0)
        y_pos = torch.clamp(y10[:, 4:10], 0.0, 1.0)
        active = torch.cat([y_signed, y_pos], dim=1)

        raw = active * 32767.0
        zeros = torch.zeros((x.shape[0], 2), dtype=x.dtype, device=x.device)
        return torch.cat([raw, zeros], dim=1)  # [batch, 12]


wrapper = RidgeDecoderWrapper(x_mean, x_scale, W, b).eval()

In [ ]:
final_onnx_path = os.path.join(OUT_DIR, "model.onnx")
dummy = torch.randn(1, FEATURE_DIM, dtype=torch.float32)

torch.onnx.export(
    wrapper,
    dummy,
    final_onnx_path,
    input_names=["features"],
    output_names=["controller"],
    dynamic_axes={
        "features": {0: "batch"},
        "controller": {0: "batch"},
    },
    opset_version=17,
)

print("Saved final ONNX to:", final_onnx_path)

In [ ]:
sess = ort.InferenceSession(final_onnx_path, providers=["CPUExecutionProvider"])

x_test = np.random.randn(4, FEATURE_DIM).astype(np.float32)
onnx_out = sess.run(None, {"features": x_test})[0]

with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(x_test)).numpy()

print("onnx_out shape:", onnx_out.shape)
print("torch_out shape:", torch_out.shape)
max_diff = np.max(np.abs(onnx_out - torch_out))
# float32 rounding through large matmul, amplified by x32767 output scaling
print(f"max abs diff: {max_diff:.2e}")
assert max_diff < 0.1, f"ONNX roundtrip failed! max_diff={max_diff}"
print("ROUNDTRIP OK")

In [ ]:
meta = {
    "data_dir": DATA_DIR,
    "files": [os.path.basename(f) for f in FILES],
    "dataset_key": DATASET_KEY,
    "total_channels": TOTAL_CHANNELS,
    "encoded_channels": ENCODED_CHANNELS,
    "ground_truth_channels": GT_CHANNELS,
    "predicted_channels": list(range(10)),
    "constant_zero_channels": [10, 11],
    "sample_rate_hz": FS,
    "bin_size_raw_frames": BIN_SIZE,
    "bin_duration_ms": 1000.0 * BIN_SIZE / FS,
    "context_bins": CONTEXT_BINS,
    "context_duration_ms": 1000.0 * BIN_SIZE * CONTEXT_BINS / FS,
    "feature_mode": FEATURE_MODE,
    "onnx_input_shape": [1, FEATURE_DIM],
    "onnx_output_shape": [1, GT_CHANNELS],
    "best_alpha": float(best_alpha),
    "notes": (
        f"Windowed ridge decoder. Input is {CONTEXT_BINS} bins x {ENCODED_CHANNELS} channels "
        f"{'x 2 (mean+RMS) ' if FEATURE_MODE == 'mean_rms' else ''}"
        f"flattened to {FEATURE_DIM}. "
        "Outputs gt_0..gt_9 as clipped controller-like values in the original scale; "
        "gt_10 and gt_11 are hard-set to zero."
    ),
    "synapse_app_parameters": {
        "model_name": "model",
        "input_channels": ENCODED_CHANNELS,
        "frames_per_step": BIN_SIZE,
        "feature_window": CONTEXT_BINS,
        "feature_mode": FEATURE_MODE,
    }
}

meta_path = os.path.join(OUT_DIR, "decoder_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta to:", meta_path)
print("Main file for deployment:", final_onnx_path)

In [ ]:
import json

with open("/content/decoder_outputs_ridge/decoder_meta.json", "r") as f:
    meta = json.load(f)

fs = meta["sample_rate_hz"]                          # 32000
frames_per_step = meta["synapse_app_parameters"]["frames_per_step"]  # 320
steps_per_sec = fs / frames_per_step

n_total_outputs = meta["ground_truth_channels"]      # 12
n_pred_outputs = len(meta["predicted_channels"])     # 10

for bits_per_output in [8, 16]:
    bitrate_total = steps_per_sec * n_total_outputs * bits_per_output
    bitrate_pred = steps_per_sec * n_pred_outputs * bits_per_output
    print(f"{bits_per_output}-bit, 12 outputs: {bitrate_total:.1f} bps ({bitrate_total/1000:.2f} kbps)")
    print(f"{bits_per_output}-bit, 10 learned outputs: {bitrate_pred:.1f} bps ({bitrate_pred/1000:.2f} kbps)")

8-bit, 12 outputs: 9600.0 bps (9.60 kbps)
8-bit, 10 learned outputs: 8000.0 bps (8.00 kbps)
16-bit, 12 outputs: 19200.0 bps (19.20 kbps)
16-bit, 10 learned outputs: 16000.0 bps (16.00 kbps)


In [ ]:
import numpy as np

def quantize_outputs(y, bits=8):
    """
    y: [T, 12] or [T, 10] decoder outputs in original controller scale.
       Expected ranges from notebook:
       - channels 0:4 are signed, roughly [-32767, 32767]
       - channels 4:10 are nonnegative, roughly [0, 32767]
    Returns integer-coded array.
    """
    y = np.asarray(y).copy()

    if y.shape[1] >= 4:
        y[:, :4] = np.clip(y[:, :4], -32767, 32767)
    if y.shape[1] > 4:
        y[:, 4:] = np.clip(y[:, 4:], 0, 32767)

    levels = 2 ** bits

    q = np.zeros_like(y, dtype=np.int32)

    # signed channels -> map [-32767,32767] to [0, levels-1]
    q[:, :4] = np.round((y[:, :4] + 32767) * (levels - 1) / (2 * 32767)).astype(np.int32)

    # nonnegative channels -> map [0,32767] to [0, levels-1]
    if y.shape[1] > 4:
        q[:, 4:] = np.round(y[:, 4:] * (levels - 1) / 32767).astype(np.int32)

    q = np.clip(q, 0, levels - 1)
    return q


def entropy_bits(x, levels=None):
    x = np.asarray(x).astype(np.int64).ravel()
    if levels is None:
        levels = int(x.max()) + 1
    counts = np.bincount(x, minlength=levels).astype(np.float64)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())


def estimate_entropy_bitrate(y_pred, steps_per_sec=100, bits=8):
    """
    y_pred: [T, C] predictions in original controller scale
    Returns total entropy rate and per-channel entropies.
    """
    q = quantize_outputs(y_pred, bits=bits)
    levels = 2 ** bits

    per_channel_entropy = [entropy_bits(q[:, c], levels=levels) for c in range(q.shape[1])]
    bits_per_step = float(np.sum(per_channel_entropy))
    bits_per_sec = bits_per_step * steps_per_sec

    return {
        "bits_per_step": bits_per_step,
        "bits_per_sec": bits_per_sec,
        "kbps": bits_per_sec / 1000.0,
        "per_channel_entropy_bits": per_channel_entropy,
    }

# Example:
# y_pred should be your decoder outputs in original scale, shape [T, 10] or [T, 12]
# result = estimate_entropy_bitrate(y_pred, steps_per_sec=100, bits=8)
# print(result)

In [ ]:
import onnx

model_path = "/content/decoder_outputs_ridge/model.onnx"
patched_path = "/content/decoder_outputs_ridge/model_ir9.onnx"

m = onnx.load(model_path)

print("Before:")
print("  ir_version =", m.ir_version)
print("  opsets =", [(op.domain, op.version) for op in m.opset_import])

# Device ORT 1.16 supports IR version 9
m.ir_version = 9

onnx.save(m, patched_path)

m2 = onnx.load(patched_path)
print("After:")
print("  ir_version =", m2.ir_version)
print("  opsets =", [(op.domain, op.version) for op in m2.opset_import])

Before:
  ir_version = 10
  opsets = [('', 17)]
After:
  ir_version = 9
  opsets = [('', 17)]
